<a href="https://colab.research.google.com/github/sparshdubey95/AI-Legal-Doc-Analyzer/blob/main/tsp_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install numpy scipy matplotlib deap tsplib95

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 17.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 13.7 MB/s  0:00:00
  Attempting uninstall: wrapt
    Found existing installation: wrapt 2.5.2
    Uninstalling wrapt-2.5.2:
      Successfully uninstalled wrapt-2.5.2
  Attempting uninstall: tabulate
    Found existing installation: tabulate 0.9.0
    Uninstalling tabulate-0.9.0:
      Successfully uninstalled tabulate-0.9.0
  Attempting uninstall: networkx
    Found existing installation: networkx 3.7
    Uninstalling networkx-3.7:
      Successfully uninstalled networkx-3.7
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [deap]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scikit-image 0.25.2 requires networkx>=3.0, but you have networkx 2.8.8 which is incompatible.
spopt 0.7.0 requires networkx>=3.2, but you have networkx 2.8

In [ ]:
"""
Problem 5: Travelling Salesman Problem (TSP) - Crossover Operator Study
Dataset: ulysses22 from TSPLIB (Optimal tour length = 7013)

This script compares 3 Crossover Operators under identical experimental conditions:
  1. Edge Recombination (ER) - Edge-based operator
  2. Partially Mapped Crossover (PMX) - Position-based operator
  3. Order Crossover (OX) - Order-based operator

Metrics Tracked:
  - Best Tour Distance across 1000 generations
  - Edge Heritability Metric per generation
  - Mann-Whitney U test p-values & Vargha-Delaney A12 effect sizes

Author: Evolutionary Computation Lab Project
Date: November 2026
"""

import math
import random
import csv
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from deap import base, creator, tools

# -----------------------------------------------------------------------------
# 1. TSPLIB ulysses22 Dataset & GEO Distance Computation
# -----------------------------------------------------------------------------
ULYSSES22_TSPLIB_DATA = """
NAME: ulysses22.tsp
TYPE: TSP
COMMENT: Odyssey of Homer (Groetschel/Padberg)
DIMENSION: 22
EDGE_WEIGHT_TYPE: GEO
NODE_COORD_SECTION
1 38.24 20.42
2 39.57 26.15
3 40.56 25.32
4 36.26 23.12
5 33.48 26.54
6 37.56 25.32
7 38.42 20.27
8 37.03 22.49
9 35.18 19.03
10 36.28 16.53
11 37.10 14.40
12 39.15 15.21
13 38.15 15.35
14 37.23 15.34
15 36.47 15.13
16 38.00 13.04
17 37.30 11.02
18 40.21 13.48
19 40.13 14.28
20 40.10 14.09
21 40.30 13.03
22 39.41 10.12
EOF
"""

def parse_ulysses22_coords(data_str):
    """Parses node coordinates from embedded TSPLIB text string."""
    coords = []
    lines = data_str.strip().split('\n')
    in_section = False
    for line in lines:
        line = line.strip()
        if line == "NODE_COORD_SECTION":
            in_section = True
            continue
        if line == "EOF" or not line:
            break
        if in_section:
            parts = line.split()
            idx = int(parts[0]) - 1
            lat = float(parts[1])
            lon = float(parts[2])
            coords.append((lat, lon))
    return coords

def geo_distance(coord1, coord2):
    """Computes exact TSPLIB GEO distance between two coordinates."""
    PI = 3.141592
    RR = 6378.388

    lat1, lon1 = coord1
    lat2, lon2 = coord2

    # Convert deg.min to radians as specified by TSPLIB documentation
    deg_lat1 = int(lat1)
    min_lat1 = lat1 - deg_lat1
    rad_lat1 = PI * (deg_lat1 + 5.0 * min_lat1 / 3.0) / 180.0

    deg_lon1 = int(lon1)
    min_lon1 = lon1 - deg_lon1
    rad_lon1 = PI * (deg_lon1 + 5.0 * min_lon1 / 3.0) / 180.0

    deg_lat2 = int(lat2)
    min_lat2 = lat2 - deg_lat2
    rad_lat2 = PI * (deg_lat2 + 5.0 * min_lat2 / 3.0) / 180.0

    deg_lon2 = int(lon2)
    min_lon2 = lon2 - deg_lon2
    rad_lon2 = PI * (deg_lon2 + 5.0 * min_lon2 / 3.0) / 180.0

    q1 = math.cos(rad_lon1 - rad_lon2)
    q2 = math.cos(rad_lat1 - rad_lat2)
    q3 = math.cos(rad_lat1 + rad_lat2)
    d = int(RR * math.acos(0.5 * ((1.0 + q1) * q2 - (1.0 - q1) * q3)) + 1.0)
    return d

def compute_distance_matrix(coords):
    """Precomputes N x N distance matrix for ulysses22."""
    n = len(coords)
    dist_matrix = np.zeros((n, n), dtype=int)
    for i in range(n):
        for j in range(n):
            if i != j:
                dist_matrix[i][j] = geo_distance(coords[i], coords[j])
    return dist_matrix

# Load problem data
COORDS = parse_ulysses22_coords(ULYSSES22_TSPLIB_DATA)
N_CITIES = len(COORDS)
DIST_MATRIX = compute_distance_matrix(COORDS)

# -----------------------------------------------------------------------------
# 2. TSP Evaluation & Edge Heritability Metrics
# -----------------------------------------------------------------------------
def eval_tsp(individual):
    """Calculates total tour length of a permutation individual."""
    tour_len = 0
    n = len(individual)
    for i in range(n):
        u = individual[i]
        v = individual[(i + 1) % n]
        tour_len += DIST_MATRIX[u][v]
    return (tour_len,)

def get_tour_edges(individual):
    """Extracts set of undirected edges (u, v) from a tour permutation."""
    n = len(individual)
    edges = set()
    for i in range(n):
        u = individual[i]
        v = individual[(i + 1) % n]
        edges.add(tuple(sorted((u, v))))
    return edges

def calculate_edge_heritability(child, p1, p2):
    """Computes fraction of child's edges inherited from parent 1 or parent 2."""
    c_edges = get_tour_edges(child)
    p_edges = get_tour_edges(p1).union(get_tour_edges(p2))
    return len(c_edges.intersection(p_edges)) / float(len(child))

# -----------------------------------------------------------------------------
# 3. Custom Operators (Edge Recombination & Inversion Mutation)
# -----------------------------------------------------------------------------
def cx_edge_recombination(ind1, ind2):
    """
    Edge Recombination Crossover (ER): Preserves direct edge adjacencies.
    Modifies ind1 and ind2 in-place to return two valid permutation offspring.
    """
    n = len(ind1)

    def generate_er_child(parent_a, parent_b):
        # Build adjacency map
        edge_map = {i: set() for i in range(n)}
        for p in (parent_a, parent_b):
            for i in range(n):
                u = p[i]
                v = p[(i + 1) % n]
                edge_map[u].add(v)
                edge_map[v].add(u)

        curr = random.choice([parent_a[0], parent_b[0]])
        child = [curr]
        visited = {curr}

        while len(child) < n:
            # Remove current city from all neighbor lists
            for neighbors in edge_map.values():
                neighbors.discard(curr)

            unvisited_neighbors = [node for node in edge_map[curr] if node not in visited]

            if unvisited_neighbors:
                # Select neighbor with smallest remaining degree (tie-break randomly)
                min_deg = min(len(edge_map[node]) for node in unvisited_neighbors)
                candidates = [node for node in unvisited_neighbors if len(edge_map[node]) == min_deg]
                nxt = random.choice(candidates)
            else:
                # Random tie-break if no adjacent neighbors exist
                unvisited = [node for node in range(n) if node not in visited]
                nxt = random.choice(unvisited)

            curr = nxt
            child.append(curr)
            visited.add(curr)

        return child

    c1 = generate_er_child(ind1, ind2)
    c2 = generate_er_child(ind2, ind1)

    ind1[:], ind2[:] = c1, c2
    return ind1, ind2

def mut_inversion(individual):
    """
    Inversion Mutation: Reverses a random sub-slice of the permutation array.
    """
    n = len(individual)
    i, j = sorted(random.sample(range(n), 2))
    individual[i:j] = list(reversed(individual[i:j]))
    return (individual,)

# -----------------------------------------------------------------------------
# 4. DEAP Global Setup
# -----------------------------------------------------------------------------
# Declare types once at global scope to prevent duplicate type warnings
if not hasattr(creator, "FitnessMin"):
    creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
if not hasattr(creator, "Individual"):
    creator.create("Individual", list, fitness=creator.FitnessMin)

def build_toolbox(cx_operator_name):
    """Factory creating a configured DEAP toolbox for a specific crossover operator."""
    toolbox = base.Toolbox()
    toolbox.register("indices", random.sample, range(N_CITIES), N_CITIES)
    toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.indices)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("evaluate", eval_tsp)
    toolbox.register("select", tools.selTournament, tournsize=2)
    toolbox.register("mutate", mut_inversion)

    if cx_operator_name == "ER":
        toolbox.register("mate", cx_edge_recombination)
    elif cx_operator_name == "PMX":
        toolbox.register("mate", tools.cxPartialyMatched)
    elif cx_operator_name == "OX":
        toolbox.register("mate", tools.cxOrdered)
    else:
        raise ValueError(f"Unknown crossover operator: {cx_operator_name}")

    return toolbox

# -----------------------------------------------------------------------------
# 5. Experimental Runner Loop
# -----------------------------------------------------------------------------
def run_single_ea(cx_name, seed, pop_size=100, n_gens=1000, cx_pb=0.8, mut_pb=0.2, n_elites=2):
    """Runs one independent GA trial and logs convergence and heritability metrics."""
    random.seed(seed)
    np.random.seed(seed)

    toolbox = build_toolbox(cx_name)
    pop = toolbox.population(n=pop_size)

    # Initial Evaluation
    for ind in pop:
        ind.fitness.values = toolbox.evaluate(ind)

    best_distance_history = []
    heritability_history = []

    for gen in range(n_gens):
        # 1. Elitism: Preserve best n_elites tours
        elites = [toolbox.clone(ind) for ind in tools.selBest(pop, n_elites)]

        # 2. Tournament Selection
        offspring = toolbox.select(pop, pop_size - n_elites)
        offspring = [toolbox.clone(ind) for ind in offspring]

        # 3. Crossover & Heritability Metric Capture
        gen_heritability_vals = []
        for i in range(0, len(offspring) - 1, 2):
            if random.random() < cx_pb:
                p1 = toolbox.clone(offspring[i])
                p2 = toolbox.clone(offspring[i + 1])

                toolbox.mate(offspring[i], offspring[i + 1])

                # Calculate heritability for offspring
                h1 = calculate_edge_heritability(offspring[i], p1, p2)
                h2 = calculate_edge_heritability(offspring[i + 1], p1, p2)
                gen_heritability_vals.extend([h1, h2])

                del offspring[i].fitness.values
                del offspring[i + 1].fitness.values

        # 4. Inversion Mutation
        for ind in offspring:
            if random.random() < mut_pb:
                toolbox.mutate(ind)
                del ind.fitness.values

        # 5. Evaluate Invalid Offspring
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        for ind in invalid_ind:
            ind.fitness.values = toolbox.evaluate(ind)

        # Recombine elites and offspring
        pop = elites + offspring

        # Record metrics for current generation
        best_ind = tools.selBest(pop, 1)[0]
        best_distance_history.append(best_ind.fitness.values[0])

        mean_h = np.mean(gen_heritability_vals) if gen_heritability_vals else 0.0
        heritability_history.append(mean_h)

    final_best_dist = best_distance_history[-1]
    return final_best_dist, best_distance_history, heritability_history

# -----------------------------------------------------------------------------
# 6. Statistical Test Helpers
# -----------------------------------------------------------------------------
def vargha_delaney_a12(sample1, sample2):
    """
    Calculates the Vargha-Delaney A12 effect size statistic.
    Interpretation: A12 < 0.5 indicates sample1 tends to be smaller (better for minimization).
    """
    m = len(sample1)
    n = len(sample2)
    r = stats.rankdata(np.concatenate([sample1, sample2]))
    r1 = np.sum(r[:m])
    a = (r1 / m - (m + 1) / 2.0) / n
    return a

# -----------------------------------------------------------------------------
# 7. Main Execution Flow
# -----------------------------------------------------------------------------
def main():
    print("=" * 75)
    print("  TSP CROSSOVER OPERATOR COMPARISON: ulysses22 (TSPLIB)")
    print("  Target Optimal Distance: 7013")
    print("=" * 75)

    OPERATORS = ["ER", "PMX", "OX"]
    N_RUNS = 30
    N_GENS = 1000
    POP_SIZE = 100
    BASE_SEED = 2026

    results_data = {op: [] for op in OPERATORS}
    convergence_histories = {op: [] for op in OPERATORS}
    heritability_histories = {op: [] for op in OPERATORS}

    # Execute 30 Independent Runs for Each Operator
    for op in OPERATORS:
        print(f"\n[+] Executing {N_RUNS} independent runs for operator: {op} ...")
        for r in range(N_RUNS):
            seed = BASE_SEED + r
            final_dist, dist_hist, h_hist = run_single_ea(
                cx_name=op, seed=seed, pop_size=POP_SIZE, n_gens=N_GENS
            )
            results_data[op].append(final_dist)
            convergence_histories[op].append(dist_hist)
            heritability_histories[op].append(h_hist)

        mean_final = np.mean(results_data[op])
        median_final = np.median(results_data[op])
        print(f"    -> Completed {op}: Median = {median_final:.1f}, Mean = {mean_final:.1f}")

    # Compute Aggregate Histories across Runs
    mean_convergence = {op: np.mean(convergence_histories[op], axis=0) for op in OPERATORS}
    mean_heritability = {op: np.mean(heritability_histories[op], axis=0) for op in OPERATORS}

    # Statistical Evaluation
    print("\n" + "=" * 75)
    print("  STATISTICAL EVALUATION & EFFECT SIZE (30 Independent Runs)")
    print("=" * 75)

    summary_rows = []
    for op in OPERATORS:
        arr = np.array(results_data[op])
        summary_rows.append({
            "Operator": op,
            "Median": np.median(arr),
            "Mean": np.mean(arr),
            "Std": np.std(arr),
            "Min": np.min(arr),
            "Max": np.max(arr)
        })

    # Print Formatted Table
    header = f"{'Operator':<10} | {'Median':<9} | {'Mean':<9} | {'Std':<8} | {'Min':<8} | {'Max':<8}"
    print(header)
    print("-" * len(header))
    for r in summary_rows:
        print(f"{r['Operator']:<10} | {r['Median']:<9.1f} | {r['Mean']:<9.1f} | {r['Std']:<8.1f} | {r['Min']:<8.0f} | {r['Max']:<8.0f}")

    # Mann-Whitney U Tests & A12 Effect Sizes (Comparing ER against PMX and OX)
    print("\n--- Hypothesis Testing (Mann-Whitney U & Vargha-Delaney A12) ---")
    er_vals = results_data["ER"]
    for comp_op in ["PMX", "OX"]:
        comp_vals = results_data[comp_op]
        stat_val, p_val = stats.mannwhitneyu(er_vals, comp_vals, alternative="two-sided")
        a12 = vargha_delaney_a12(er_vals, comp_vals)
        print(f"ER vs {comp_op:3s} | p-value: {p_val:.4e} | A12 Effect Size: {a12:.4f} "
              f"({'Statistically Significant' if p_val < 0.05 else 'Not Significant'})")

    # Export CSV Summary File
    with open("tsp_results_summary.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Operator", "Median", "Mean", "Std", "Min", "Max"])
        for r in summary_rows:
            writer.writerow([r["Operator"], r["Median"], r["Mean"], r["Std"], r["Min"], r["Max"]])
    print("\n[✓] Results exported to 'tsp_results_summary.csv'")

    # -------------------------------------------------------------------------
    # 8. Publication-Quality Plot Generation
    # -------------------------------------------------------------------------
    gens_axis = np.arange(1, N_GENS + 1)
    colors = {"ER": "#1f77b4", "PMX": "#ff7f0e", "OX": "#2ca02c"}

    # Figure 1: Best Tour Length Convergence Curves
    plt.figure(figsize=(9, 5.5), dpi=300)
    for op in OPERATORS:
        plt.plot(gens_axis, mean_convergence[op], label=f"{op} Crossover", color=colors[op], linewidth=2.0)
    plt.axhline(y=7013, color='black', linestyle='--', linewidth=1.5, label="TSPLIB Optimal (7013)")
    plt.xlabel("Generation", fontsize=12)
    plt.ylabel("Mean Best Tour Distance", fontsize=12)
    plt.title("ulysses22: Tour Distance Convergence Comparison", fontsize=14, fontweight='bold')
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.savefig("tsp_convergence.png")
    plt.close()
    print("[✓] Plot saved: 'tsp_convergence.png'")

    # Figure 2: Mean Edge Heritability vs Generation
    plt.figure(figsize=(9, 5.5), dpi=300)
    for op in OPERATORS:
        plt.plot(gens_axis, mean_heritability[op], label=f"{op} Crossover", color=colors[op], linewidth=2.0)
    plt.xlabel("Generation", fontsize=12)
    plt.ylabel(r"Mean Edge Heritability $H(C \mid P_1, P_2)$", fontsize=12)
    plt.title("ulysses22: Edge Heritability Across Generations", fontsize=14, fontweight='bold')
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.savefig("tsp_heritability.png")
    plt.close()
    print("[✓] Plot saved: 'tsp_heritability.png'")

    # Figure 3: Boxplot of Final Tour Length Distributions
    plt.figure(figsize=(7, 5), dpi=300)
    data_to_plot = [results_data[op] for op in OPERATORS]
    bp = plt.boxplot(data_to_plot, patch_artist=True, tick_labels=OPERATORS)
    for patch, op in zip(bp['boxes'], OPERATORS):
        patch.set_facecolor(colors[op])
        patch.set_alpha(0.7)
    plt.axhline(y=7013, color='black', linestyle='--', linewidth=1.5, label="Optimal (7013)")
    plt.ylabel("Final Best Tour Distance", fontsize=12)
    plt.title("ulysses22: Final Distance Distribution (30 Runs)", fontsize=13, fontweight='bold')
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig("tsp_boxplot.png")
    plt.close()
    print("[✓] Plot saved: 'tsp_boxplot.png'")

    print("\n[+] All tasks complete! You are ready to embed the results into your PDF report.")

if __name__ == "__main__":
    main()

  TSP CROSSOVER OPERATOR COMPARISON: ulysses22 (TSPLIB)
  Target Optimal Distance: 7013

[+] Executing 30 independent runs for operator: ER ...
    -> Completed ER: Median = 4521.0, Mean = 4563.0

[+] Executing 30 independent runs for operator: PMX ...
    -> Completed PMX: Median = 4544.0, Mean = 4589.7

[+] Executing 30 independent runs for operator: OX ...
